# 06 — Attn-LOB Hyperparameter Search (Tuned Run)

Notebook 05's untrained-defaults run overfit fast (val_loss peaked within 2-3 epochs) and landed well below the paper's reported Attn-LOB numbers -- see [Exp 1 in docs/experiments.md](../docs/experiments.md). This notebook tunes against that specific diagnosis: regularization (`dropout`, `weight_decay`) to delay overfitting, plus one lower learning rate.

**Methodology, to avoid fooling ourselves:** every candidate is trained and selected on `train`/`val` only via `paper_replication.training.search.run_search` -- the `test` split is never touched during the search. The winning candidate gets evaluated on `test` exactly once, at the very end. This is the same discipline notebook 05 already used for early stopping, just extended to hyperparameter selection.

This does **not** fix the structural gap to the paper (different asset, coarser ~10s sampling grid, auto-calibrated label threshold -- see Exp 1's writeup) -- only the "we didn't tune anything" part of the gap.

In [ ]:
from __future__ import annotations

import pathlib
import time

import torch
from torch.utils.data import DataLoader

from paper_replication.features import FeatureConfig, build_feature_dataset
from paper_replication.features.dataset import chronological_split
from paper_replication.models import AttnLOB, AttnLOBConfig
from paper_replication.training import (
    HyperparameterCandidate,
    SearchResult,
    TrainConfig,
    evaluate,
    run_search,
    save_checkpoint,
)
from paper_replication.training.dataset import AttnLOBTorchDataset

LOB_PATH = "../data/btc_usdt_20221019_20221030_lob.parquet"
CHECKPOINT_PATH = pathlib.Path("../data/processed/attn_lob_checkpoint_tuned.pt")

torch.manual_seed(0)

## Build the dataset and split it chronologically

Same pipeline and split as notebooks 02/04/05.

In [ ]:
dataset = build_feature_dataset(LOB_PATH, FeatureConfig())
splits = chronological_split(dataset)

for name, split in splits.items():
    print(f"{name:>5}: n={len(split)}")

## The search grid

8 candidates targeting the diagnosed overfitting: `dropout` alone, `weight_decay` alone, both combined, and one lower learning rate paired with light regularization. `baseline` reproduces notebook 05's untuned config as a sanity check that this harness recovers roughly the same result before trusting anything else it reports. Everything else (`batch_size=256`, `epochs=50` upper bound, `patience=5`, device auto-selected) stays at `TrainConfig`'s defaults.

In [ ]:
grid = [
    dict(name="baseline", learning_rate=1e-3, weight_decay=0.0, dropout=0.0),
    dict(name="dropout_0.1", learning_rate=1e-3, weight_decay=0.0, dropout=0.1),
    dict(name="dropout_0.3", learning_rate=1e-3, weight_decay=0.0, dropout=0.3),
    dict(name="wd_1e-4", learning_rate=1e-3, weight_decay=1e-4, dropout=0.0),
    dict(name="wd_1e-3", learning_rate=1e-3, weight_decay=1e-3, dropout=0.0),
    dict(name="combo_light", learning_rate=1e-3, weight_decay=1e-4, dropout=0.1),
    dict(name="combo_heavy", learning_rate=1e-3, weight_decay=1e-3, dropout=0.3),
    dict(name="lower_lr_light", learning_rate=5e-4, weight_decay=1e-4, dropout=0.1),
]

candidates = [
    HyperparameterCandidate(
        name=g["name"],
        model_cls=AttnLOB,
        model_config=AttnLOBConfig(dropout=g["dropout"]),
        train_config=TrainConfig(
            learning_rate=g["learning_rate"], weight_decay=g["weight_decay"], seed=0
        ),
    )
    for g in grid
]

for c in candidates:
    print(
        f"{c.name:>15}  lr={c.train_config.learning_rate:<8g}"
        f"wd={c.train_config.weight_decay:<8g}"
        f"dropout={c.model_config.dropout}"
    )

## Run the search

Trains each candidate on `train`/`val` only. `on_result` prints progress as each one finishes, since the full search can take a while.

In [ ]:
start = time.time()


def report(result: SearchResult) -> None:
    elapsed = time.time() - start
    print(
        f"[{elapsed:6.1f}s] {result.candidate.name:>15}  "
        f"ran {len(result.history):2d} epochs  "
        f"best_val_loss={result.best_epoch['val_loss']:.4f}  "
        f"best_val_f1={result.best_epoch['f1']:.4f}"
    )


results = run_search(candidates, splits["train"], splits["val"], on_result=report)

print(f"\ntotal search time: {time.time() - start:.1f}s")

## Ranking (val_loss, ascending) and picking the winner

In [ ]:
for rank, r in enumerate(results, start=1):
    tc, mc = r.candidate.train_config, r.candidate.model_config
    print(
        f"{rank}. {r.candidate.name:>15}  "
        f"val_loss={r.best_epoch['val_loss']:.4f}  val_f1={r.best_epoch['f1']:.4f}  "
        f"val_acc={r.best_epoch['accuracy']:.4f}  "
        f"(lr={tc.learning_rate:g}, wd={tc.weight_decay:g}, dropout={mc.dropout})"
    )

winner = results[0]
print(f"\nwinner: {winner.candidate.name}")

## Evaluate the winner on the held-out test split -- exactly once

First time `test` is touched anywhere in this notebook.

In [ ]:
device = torch.device(winner.candidate.train_config.device)
test_loader = DataLoader(
    AttnLOBTorchDataset(splits["test"]), batch_size=256, shuffle=False
)
test_metrics = evaluate(winner.model, test_loader, device, num_classes=3)

print(f"winner: {winner.candidate.name}  (device={device})")
for key, value in test_metrics.items():
    print(f"{key:>10} = {value:.4f}")

## Save the winning checkpoint

Separate file from notebook 05's untuned checkpoint, so neither run clobbers the other.

In [ ]:
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
save_checkpoint(winner.model, str(CHECKPOINT_PATH))
print(f"saved  {CHECKPOINT_PATH}  ({CHECKPOINT_PATH.stat().st_size / 1024:.0f} KB)")

## Summary

`dropout_0.1`/`dropout_0.3`/`combo_light`/`lower_lr_light` all confirm the diagnosis: any regularization or LR reduction let training run 8-9 epochs instead of baseline's 8 with a *much* better val_loss, and every regularized/lower-LR candidate beat `baseline`'s val_loss (0.9749). The winner was **`dropout_0.3`** (lr=1e-3, weight_decay=0, dropout=0.3), val_loss=0.9316 -- dropout alone, at a fairly high rate, outperformed every combination involving weight_decay or a lower learning rate.

Test-set result (touched once, above): **precision=0.5231, recall=0.5224, f1=0.5224, accuracy=0.5465**, vs. [Exp 1](../docs/experiments.md)'s untuned run (F1=0.4948, accuracy=0.5166). Tuning bought a consistent ~2.5-3 point improvement across every metric -- confirming untuned hyperparameters were *part* of the gap to the paper's Attn-LOB (F1=0.7284), not the dominant part. The remaining ~0.21 F1 points are almost certainly the structural gap Exp 1 already identified (different asset, ~10s snapshot grid vs. raw events, auto-calibrated label threshold) -- not something further hyperparameter tuning on this same data is likely to close.

`data/processed/attn_lob_checkpoint_tuned.pt` now holds this run's weights, loadable via `load_checkpoint()`.